# Day 2: Advanced SQL (Querying, Joining, & Modifying)

Yesterday we learned the basics (`CREATE`, `INSERT`, `SELECT`, `WHERE`). Today, we'll dive deeper into modifying data and running more powerful queries to combine data from multiple tables.

### 1. Setup: Re-connect and Create Tables

We'll re-connect to our `db/company.db`. We'll also create a second table, `departments`, so we can learn how to `JOIN` them.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('db/company.db')
c = conn.cursor()

c.execute("DROP TABLE IF EXISTS employees")
c.execute("DROP TABLE IF EXISTS departments")

c.execute("""CREATE TABLE employees (
    id INTEGER PRIMARY KEY, 
    name TEXT NOT NULL, 
    dept_id INTEGER, 
    salary REAL
)""")

c.execute("""CREATE TABLE departments (
    dept_id INTEGER PRIMARY KEY, 
    dept_name TEXT
)""")

print("Tables created.")

Tables created.


### 2. Task 1: Insert Data into Both Tables

In [2]:
depts = [(1, 'Engineering'), (2, 'Sales'), (3, 'HR')]
c.executemany("INSERT INTO departments (dept_id, dept_name) VALUES (?, ?)", depts)

emps = [
    ('Alice Smith', 1, 90000),
    ('Bob Johnson', 2, 65000),
    ('Charlie Brown', 1, 110000),
    ('David Lee', 2, 70000),
    ('Eva Grant', 3, 55000)
]
c.executemany("INSERT INTO employees (name, dept_id, salary) VALUES (?, ?, ?)", emps)

conn.commit()
print("Data inserted into both tables.")

Data inserted into both tables.


### 3. Core Concept: JOINs

This is the most powerful concept in SQL. A `JOIN` clause is used to combine rows from two or more tables based on a related column between them.

We have `dept_id` in both tables. We can use it to `JOIN` them and get the department *name* for each employee.

In [3]:
print("--- Employee Names and Department Names ---")
query = """
SELECT e.name, d.dept_name, e.salary
FROM employees AS e
INNER JOIN departments AS d ON e.dept_id = d.dept_id
"""

c.execute(query)
joined_data = c.fetchall()

for row in joined_data:
    print(row)

--- Employee Names and Department Names ---
('Alice Smith', 'Engineering', 90000.0)
('Bob Johnson', 'Sales', 65000.0)
('Charlie Brown', 'Engineering', 110000.0)
('David Lee', 'Sales', 70000.0)
('Eva Grant', 'HR', 55000.0)


### 4. Task 2: UPDATE Data

Let's give Alice a raise. The `UPDATE` statement modifies existing records.

In [4]:
c.execute("UPDATE employees SET salary = 95000 WHERE name = 'Alice Smith'")
conn.commit()

print("Gave Alice a raise.")
c.execute("SELECT * FROM employees WHERE name = 'Alice Smith'")
print(c.fetchone())

Gave Alice a raise.
(1, 'Alice Smith', 1, 95000.0)


### 5. Task 3: DELETE Data

Let's say Eva leaves the company. `DELETE` removes records.

In [5]:
c.execute("DELETE FROM employees WHERE name = 'Eva Grant'")
conn.commit()

print("Record for Eva Grant deleted.")
c.execute("SELECT * FROM employees WHERE name = 'Eva Grant'")
print(c.fetchall()) 

Record for Eva Grant deleted.
[]


### 6. Task 4: Aggregations (GROUP BY)

`GROUP BY` is the SQL equivalent of `pandas.groupby()`. It's used with aggregate functions like `COUNT()`, `AVG()`, `SUM()`, `MAX()`, `MIN()`.

In [6]:
print("--- Average Salary per Department ---")
query = """
SELECT d.dept_name, AVG(e.salary) AS average_salary
FROM employees AS e
INNER JOIN departments AS d ON e.dept_id = d.dept_id
GROUP BY d.dept_name
"""

c.execute(query)
avg_salaries = c.fetchall()

for row in avg_salaries:
    print(f"{row[0]}: ${row[1]:,.2f}")

--- Average Salary per Department ---
Engineering: $102,500.00
Sales: $67,500.00


### 7. Task 5: ORDER BY

`ORDER BY` is used to sort the result set. The default is ascending (`ASC`). We can use `DESC` for descending.

In [7]:
print("--- Employees Sorted by Salary (Highest First) ---")
c.execute("SELECT name, salary FROM employees ORDER BY salary DESC")
sorted_emps = c.fetchall()

for row in sorted_emps:
    print(row)

--- Employees Sorted by Salary (Highest First) ---
('Charlie Brown', 110000.0)
('Alice Smith', 95000.0)
('David Lee', 70000.0)
('Bob Johnson', 65000.0)


### 8. Cleanup

In [8]:
conn.close()
print("Connection closed.")

Connection closed.


### Day 2 Summary

Fantastic! You're now a much more powerful SQL user. You've learned how to:
1.  `UPDATE` and `DELETE` data.
2.  Create multiple related tables.
3.  Use `INNER JOIN` to combine data from two tables.
4.  Use `GROUP BY` with aggregate functions (`AVG`) to summarize data.
5.  Sort your results with `ORDER BY`.

Tomorrow, we'll start the **ETL** process by **Extracting** data from a source and **Transforming** it with `pandas`.